In [6]:
import pandas as pd
import statsmodels.api as sm    
import numpy as np
from sklearn.model_selection import train_test_split
import sys, os
sys.path.insert(0, os.path.expanduser('~/my_ukb_thesis'))
from const_paths import BASE_PATH, SAVE_DIR

In [7]:
base_path = BASE_PATH
file_exposure = os.path.join(base_path, 'group_1_clean_filtered_imputed_dataset_df.tsv') # exposure variables 
file_outcome = os.path.join(base_path, 'group_1_outcomes_df_without_qc_df.tsv') # outcome variables 
file_new_tabular = os.path.join(base_path, 'ukb_tabular_data_causal_analysis.tsv')

exposure_data = pd.read_csv(file_exposure, sep='\t') 
outcome_data = pd.read_csv(file_outcome, sep='\t')

print(f'exposure_data shape: {exposure_data.shape}')
print(f'outcome_data shape: {outcome_data.shape}')

exposure_outcome_data_raw = pd.merge(exposure_data, outcome_data, on='eid', how='inner') # merge exposure and outcome data on 'eid' column

if os.path.exists(file_new_tabular):
    cols_to_extract = ['eid', '20117-0.0','1050-0.0', '1060-0.0']
    new_tabular_data = pd.read_csv(file_new_tabular, sep='\t', usecols=cols_to_extract)
    exposure_outcome_data = pd.merge(exposure_outcome_data_raw, new_tabular_data, on='eid', how='inner') # merge the result with raw data on 'eid' column
    print(f'Merged data shape: {exposure_outcome_data.shape}')

exposure_data shape: (459142, 415)
outcome_data shape: (459142, 8)
Merged data shape: (458840, 425)


In [8]:
df_model = exposure_outcome_data.copy()

raw_cols = [
    '1289-0.0', '1309-0.0', '1349-0.0',  # Diet
    '864-0.0', '884-0.0', '22036-0.0',   # Activity
    '1050-0.0', '1060-0.0', # Sun
    '1160-0.0',             # Sleep
    '20116-0.0',            # Smoking
    '20117-0.0',            # Alcohol
    '1110-0.0', '2237-0.0', # Electronic
    '2090-0.0', '2040-0.0'  # Mental
]
existing_cols = [i for i in raw_cols if i in df_model.columns] # if column exists in df_model
for col in existing_cols:
    df_model[col] = df_model[col].apply(lambda x: np.nan if x < 0 else x)    

# 1289-0.0: vegetable intake, 1309-0.0: fruit intake
if '1289-0.0' in df_model.columns and '1309-0.0' in df_model.columns: 
    df_model['diet_total'] = (df_model['1289-0.0'] / 3) + df_model['1309-0.0'] # diet_total = vegetable / 3 + fruit    

# 1050-0.0: summer sun exposure, 1060-0.0: winter sun exposure
if '1050-0.0' in df_model.columns and '1060-0.0' in df_model.columns:
    df_model['sun_exposure_avg'] = (df_model['1050-0.0'] + df_model['1060-0.0']) / 2 # sun_exposure_avg = average of summer and winter sun exposure

# 1160-0.0: sleep duration
if '1160-0.0' in df_model.columns:
    df_model['sleep_hrs'] = df_model['1160-0.0']
    # sleep_optimal = 1 if sleep_hours <= 7 or sleep_hours => 9, else 0
    df_model['sleep_optimal'] = df_model['1160-0.0'].apply(lambda x: 1 if 7 <= x <= 9 else 0)

# 22036-0.0: at or above moderate/vigorous/walking recommendation
if '22036-0.0' in df_model.columns:
    df_model['PA_active'] = df_model['22036-0.0'] # PA_active = 1 if at or above moderate/vigorous/walking recommendation, else 0

# 20116-0.0: smoking status
if '20116-0.0' in df_model.columns:
    df_model['smoking_status'] = df_model['20116-0.0'] # smoking_status

# 20117-0.0: alcohol status
if '20117-0.0' in df_model.columns:
    df_model['alcohol_status'] = df_model['20117-0.0'] # alcohol_status

# 6138.1: education degree
if '6138.1' in df_model.columns:
    df_model['uni_degree'] = df_model['6138.1']

if '20107.1' in df_model.columns and '20107.2' in df_model.columns: # 20107.1: father history of heart disease, 20107.2: father history of stroke
    df_model['FH_cvd_f'] = ((df_model['20107.1'] == 1) | (df_model['20107.2'] == 1)).astype(int)

if '20110.1' in df_model.columns and '20110.2' in df_model.columns: # 20110.1: mother history of heart disease, 20110.2: mother history of stroke
    df_model['FH_cvd_m'] = ((df_model['20110.1'] == 1) | (df_model['20110.2'] == 1)).astype(int)

if '20111.1' in df_model.columns and '20111.2' in df_model.columns: # 20111.1: sibling history of heart disease, 20111.2: sibling history of stroke
    df_model['FH_cvd_sib'] = ((df_model['20111.1'] == 1) | (df_model['20111.2'] == 1)).astype(int)

# electronic device, mental and BMI (keep same as original)
other_cols = {
    'device_mobile':'1110-0.0', # mobile phone use
    'device_pc':'2237-0.0', # computer use
    'mental_doctor':'2090-0.0', # doctor visit
    'mental_risk':'2040-0.0', # risk taking
    'processed_meat':'1349-0.0', # processed meat 
    'BMI':'21001-0.0', # BMI
}

for new_name, old_name in other_cols.items():
    if old_name in df_model.columns:
        df_model[new_name] = df_model[old_name]

confounder = ['age_defined_baseline', 'genetic_sex','BMI','FH_cvd_f', 'FH_cvd_m', 'FH_cvd_sib', 'uni_degree'] 

# 'sun_exposure_avg','alcohol_status' missing, so not included in feature_cols for now
base_feature_cols = ['diet_total', 'sleep_hrs','PA_active',
                'device_pc', 'mental_doctor', 'mental_risk'] + confounder # feature columns

print(base_feature_cols)

dummy_feature_names = [] # create dummy variables for categorical variables: device_mobile and smoking_status

if 'device_mobile' in df_model.columns:
    mobile_dummies = pd.get_dummies(df_model['device_mobile'], prefix='mobile', dtype=float)
    if 'mobile_0' in mobile_dummies.columns: # drop the reference category (mobile_0) to avoid multicollinearity
        mobile_dummies = mobile_dummies.drop(columns=['mobile_0'])
    missing_mobile = df_model['device_mobile'].isna()
    mobile_dummies.loc[missing_mobile, :] = np.nan
    df_model = pd.concat([df_model, mobile_dummies], axis=1)
    dummy_feature_names.extend(mobile_dummies.columns.tolist())

if 'smoking_status' in df_model.columns:
    smoking_dummies = pd.get_dummies(df_model['smoking_status'], prefix='smoking', dtype=float)
    if 'smoking_0' in smoking_dummies.columns: # drop the reference category (smoking_0) to avoid multicollinearity
        smoking_dummies = smoking_dummies.drop(columns=['smoking_0'])
    missing_smoking = df_model['smoking_status'].isna()
    smoking_dummies.loc[missing_smoking, :] = np.nan
    df_model = pd.concat([df_model, smoking_dummies], axis=1)
    dummy_feature_names.extend(smoking_dummies.columns.tolist())    

if 'alcohol_status' in df_model.columns:
    df_model['alcohol_status'] = df_model['alcohol_status'].astype('Int64')
    alcohol_dummies = pd.get_dummies(df_model['alcohol_status'], prefix='alcohol', dtype=float)
    if 'alcohol_0' in alcohol_dummies.columns: # drop the reference category (alcohol_0) to avoid multicollinearity
        alcohol_dummies = alcohol_dummies.drop(columns=['alcohol_0'])
    missing_alcohol = df_model['alcohol_status'].isna()
    alcohol_dummies.loc[missing_alcohol, :] = np.nan
    df_model = pd.concat([df_model, alcohol_dummies], axis=1)
    dummy_feature_names.extend(alcohol_dummies.columns.tolist())

if 'processed_meat' in df_model.columns:
    meat_dummies = pd.get_dummies(df_model['processed_meat'], prefix='meat', dtype=float)
    if 'meat_0' in meat_dummies.columns: # drop the reference category (meat_0) to avoid multicollinearity
        meat_dummies = meat_dummies.drop(columns=['meat_0'])
    missing_meat = df_model['processed_meat'].isna()
    meat_dummies.loc[missing_meat, :] = np.nan
    df_model = pd.concat([df_model, meat_dummies], axis=1)
    dummy_feature_names.extend(meat_dummies.columns.tolist())

rename_dict = {
    'smoking_1': 'smk_prev', 'smoking_2': 'smk_curr',
    'alcohol_1': 'alc_prev', 'alcohol_2': 'alc_curr',
    'mobile_1': 'MobUse_≤1yr', 'mobile_2': 'MobUse_2-4yr', 'mobile_3': 'MobUse_5-8yr', 'mobile_4': 'MobUse_>8yr',
    'meat_1': 'ProcMeat_<1wk', 'meat_2': 'ProcMeat_1wk', 'meat_3': 'ProcMeat_2-4wk', 'meat_4': 'ProcMeat_5-6wk', 'meat_5': 'ProcMeat_daily'
}
df_model.rename(columns=rename_dict, inplace=True)
dummy_feature_names = [rename_dict.get(col, col) for col in dummy_feature_names]

feature_cols = base_feature_cols + dummy_feature_names
print(feature_cols)

['diet_total', 'sleep_hrs', 'PA_active', 'device_pc', 'mental_doctor', 'mental_risk', 'age_defined_baseline', 'genetic_sex', 'BMI', 'FH_cvd_f', 'FH_cvd_m', 'FH_cvd_sib', 'uni_degree']
['diet_total', 'sleep_hrs', 'PA_active', 'device_pc', 'mental_doctor', 'mental_risk', 'age_defined_baseline', 'genetic_sex', 'BMI', 'FH_cvd_f', 'FH_cvd_m', 'FH_cvd_sib', 'uni_degree', 'MobUse_≤1yr', 'MobUse_2-4yr', 'MobUse_5-8yr', 'MobUse_>8yr', 'smk_prev', 'smk_curr', 'alc_prev', 'alc_curr', 'ProcMeat_<1wk', 'ProcMeat_1wk', 'ProcMeat_2-4wk', 'ProcMeat_5-6wk', 'ProcMeat_daily']


In [9]:
target_col = 'def_CVD_AF_HF_AFTER'
df_final = df_model[['eid', target_col] + feature_cols]

df_temp, df_split_C = train_test_split(df_final, test_size=0.30, random_state=42, stratify=df_final[target_col]) 
df_split_B, df_split_A = train_test_split(df_temp, test_size=(0.05/0.70), random_state=42, stratify=df_temp[target_col])

print(f'split A shape 5% (Elastic net): {df_split_A.shape}')
print(f'split B shape 65% (Causal training): {df_split_B.shape}')
print(f'split C shape 30% (Causal Testing): {df_split_C.shape}')

split A shape 5% (Elastic net): (22943, 28)
split B shape 65% (Causal training): (298245, 28)
split C shape 30% (Causal Testing): (137652, 28)


In [10]:
print('\n====== the proportion of positive cases in each split ======')
for name, df_subset in zip(['Split A', 'Split B', 'Split C'], [df_split_A, df_split_B, df_split_C]):
    event_rate = df_subset[target_col].mean() * 100
    print(f"{name} Incidence of CVD: {event_rate:.3f}%")


====== the proportion of positive cases in each split ======
Split A Incidence of CVD: 8.791%
Split B Incidence of CVD: 8.790%
Split C Incidence of CVD: 8.790%


In [11]:
print(list(df_split_A.columns))

['eid', 'def_CVD_AF_HF_AFTER', 'diet_total', 'sleep_hrs', 'PA_active', 'device_pc', 'mental_doctor', 'mental_risk', 'age_defined_baseline', 'genetic_sex', 'BMI', 'FH_cvd_f', 'FH_cvd_m', 'FH_cvd_sib', 'uni_degree', 'MobUse_≤1yr', 'MobUse_2-4yr', 'MobUse_5-8yr', 'MobUse_>8yr', 'smk_prev', 'smk_curr', 'alc_prev', 'alc_curr', 'ProcMeat_<1wk', 'ProcMeat_1wk', 'ProcMeat_2-4wk', 'ProcMeat_5-6wk', 'ProcMeat_daily']


In [12]:
my_save_dir = SAVE_DIR
os.makedirs(my_save_dir, exist_ok=True)

df_split_A.to_parquet(os.path.join(my_save_dir, 'split_A_explore.parquet'), index=False)
df_split_B.to_parquet(os.path.join(my_save_dir, 'split_B_train.parquet'), index=False)
df_split_C.to_parquet(os.path.join(my_save_dir, 'split_C_test.parquet'), index=False)